## Data Cleaning and Preparation
What does this dataset do 
- this notebook cleans the data and creates a csv file that will be ready for: 

a. NASA-TLX data analysis 
- This will be the document analyzing the perceived workload of tasks done. 
    
b. Simulation data analysis
- This will be the document analyzing the statistics and the actions done by the participants during the simulation. 

Important note: 
csv files for nasa-tlx should be in 1 folder and csv files for simulation should be in a separate folder

#### Data Description


### Simulation 
| Column Name        | Description                                                  |
|--------------------|--------------------------------------------------------------|
| **timestamp** | recorded time when an occurence occured |
| **phase** | classification of drive |
| **scenario** | Type of occurence that happened|
| **speed_kmh** | recorded speed the participant was going |
| **event** | the violation or occurence type |
| **details** | Split into Violation and Location |
| **Violation** | Specific Violation acquired (?) | 
| **Location** | Exact coordinates the violation occured. This differs every time. |

### NASA TLX 
| Column Name        | Description                                                  |
|--------------------|--------------------------------------------------------------|
| **Participant**   | Participant number for tracking, acts as a unique identifier of a participant |
| **Task Name** | Type of alerts that the participant took (Baseline, AO - Audio Only, HO - Haptic Only, AH - Audio and Haptic) |
| **Mental Demand**| Mental and perceptual activity |
| **Physical Demand**| Physical activity required | 
| **Temporal Demand** | Time pressure according to the pace |
| **Performance** | Successfulness in completing the task |
| **Effort** | 
| **Frustration**| |


#### Data Collection & Methodology 
**NASATLX**
This dataset came from the NASATLX site. The participants were asked to answer the same exact questions. This was done every after drive to gauge their experience immediately after the drive. Upon completing the whole 4 drives, the csv file was generated. 

**Simulation**
This dataset came from the log tracker of the CARLA application, which records the corresponding type of drive and the violations committed. When a violation was done, the coordinates, 


#### Biases and Limitations 

#### Sources and Credits
NASA-TLX site: https://nasa-tlx-calculator.vercel.app/

**Reminders**
- make sure there are 2 separate folders > 1 each for the respective data (1 csv folder for Sim, 1 csv folder for NASATLX)

### Preprocessing

In [66]:
import pandas as pd
import numpy as np
import glob
import re
import json
 

In [67]:
# load all csv files 
# change inside "" to the folder path that contains all the simulation drives csv files. 
files = glob.glob("/Users/karencafe/Desktop/ThesisTest/SIMCSV_final/*.csv")
#/Users/karencafe/Desktop/ThesisTest/SIMCSV_final

df_simulation = pd.concat(
    (pd.read_csv(f) for f in files),
    ignore_index=True
)

display(df_simulation)


,timestamp,participant_no,phase,scenario,speed_kmh,event,details
0,1.771922e+09,17,4,START,0.00,PHASE_START,Assisted-AH
1,1.771923e+09,17,4,START,0.00,PHASE_START,Assisted-AH
2,1.771923e+09,17,4,WEBSOCKET,0.00,WS_SEND,"{""phase"": 4, ""speed"": 0.0, ""speed_limit"": 30, ..."
3,1.771923e+09,17,4,WEBSOCKET,0.00,WS_SEND,"{""phase"": 4, ""speed"": 0.0, ""speed_limit"": 30, ..."
4,1.771923e+09,17,4,WEBSOCKET,0.00,WS_SEND,"{""phase"": 4, ""speed"": 0.0, ""speed_limit"": 30, ..."
...,...,...,...,...,...,...,...
554892,1.771682e+09,11,3,WEBSOCKET,0.00,WS_SEND,"{""phase"": 3, ""speed"": 0.0, ""speed_limit"": 30, ..."
554893,1.771682e+09,11,3,WEBSOCKET,0.00,WS_SEND,"{""phase"": 3, ""speed"": 0.0, ""speed_limit"": 30, ..."
554894,1.771682e+09,11,3,WEBSOCKET,0.00,WS_SEND,"{""phase"": 3, ""speed"": 0.0, ""speed_limit"": 30, ..."
554895,1.771682e+09,11,3,STOP,674.87,PHASE_STOP,Assisted-H


In [68]:
df_simulation["timestamp"] = pd.to_numeric(df_simulation["timestamp"], errors="coerce")
df_simulation["speed_kmh"] = pd.to_numeric(df_simulation["speed_kmh"], errors="coerce")

In [69]:
# removing empty HUD alerts
df_simulation = df_simulation[
    ~(
        (df_simulation["event"] == "HUD_CUE") &
        (df_simulation["details"].str.strip() == "|")
    )
]

In [70]:
# removing speed with 0-0.1 -> too many and insignificant 
df_simulation = df_simulation[df_simulation["speed_kmh"] > 0.1]

In [71]:
df_simulation

,timestamp,participant_no,phase,scenario,speed_kmh,event,details
1300,1.771924e+09,17,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,..."
1301,1.771924e+09,17,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,..."
1302,1.771924e+09,17,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,..."
1303,1.771924e+09,17,4,WEBSOCKET,1.17,WS_SEND,"{""phase"": 4, ""speed"": 1.17, ""speed_limit"": 30,..."
1304,1.771924e+09,17,4,WEBSOCKET,1.19,WS_SEND,"{""phase"": 4, ""speed"": 1.19, ""speed_limit"": 30,..."
...,...,...,...,...,...,...,...
554840,1.771682e+09,11,3,WEBSOCKET,3.53,WS_SEND,"{""phase"": 3, ""speed"": 3.53, ""speed_limit"": 30,..."
554841,1.771682e+09,11,3,WEBSOCKET,3.53,WS_SEND,"{""phase"": 3, ""speed"": 3.53, ""speed_limit"": 30,..."
554842,1.771682e+09,11,3,WEBSOCKET,3.41,WS_SEND,"{""phase"": 3, ""speed"": 3.41, ""speed_limit"": 30,..."
554895,1.771682e+09,11,3,STOP,674.87,PHASE_STOP,Assisted-H


In [72]:
## SET SPEED_LIMIT column
def extract_speed_limit(text):
    if pd.isna(text):
        return None
    
    text = str(text)
    
    # Case 1: JSON-like
    if text.strip().startswith("{"):
        try:
            data = json.loads(text)
            return float(data.get("speed_limit"))
        except:
            return None
    
    # Case 2: Plain text
    match = re.search(r"limit\s([\d.]+)", text)
    if match:
        return float(match.group(1))
    
    return None

df_simulation["speed_limit"] = df_simulation["details"].apply(extract_speed_limit)

In [73]:
display(df_simulation[df_simulation["phase"] == 2].tail(20))

,timestamp,participant_no,phase,scenario,speed_kmh,event,details,speed_limit
520087,1.771679e+09,10,2,SPEED_LIMIT,61.60,REACTION,action=REDUCE_THROTTLE|time=1.74s|control={'th...,NaN
520088,1.771679e+09,10,2,SPEED_LIMIT,62.28,SPEED_VIOLATION,Speed 62.28 km/h > limit 30.00 km/h at Locatio...,30.0
520089,1.771679e+09,10,2,SPEED_LIMIT,37.00,REACTION,action=VIOLATION_RESOLVED|time=3.27s|reaction_...,NaN
520090,1.771679e+09,10,2,SPEED_LIMIT,38.69,REACTION,action=REDUCE_THROTTLE|time=0.10s|control={'th...,NaN
520091,1.771679e+09,10,2,SPEED_LIMIT,38.86,SPEED_VIOLATION,Speed 38.86 km/h > limit 30.00 km/h at Locatio...,30.0
520092,1.771679e+09,10,2,SPEED_LIMIT,36.94,REACTION,action=VIOLATION_RESOLVED|time=0.42s|reaction_...,NaN
520093,1.771679e+09,10,2,SPEED_LIMIT,52.57,REACTION,action=REDUCE_THROTTLE|time=2.49s|control={'th...,NaN
520094,1.771679e+09,10,2,SPEED_LIMIT,52.57,SPEED_VIOLATION,Speed 52.57 km/h > limit 30.00 km/h at Locatio...,30.0
520095,1.771679e+09,10,2,SPEED_LIMIT,35.48,REACTION,action=VIOLATION_RESOLVED|time=3.63s|reaction_...,NaN
520096,1.771679e+09,10,2,TRAFFIC_LIGHT,31.68,RED_LIGHT_VIOLATION,TrafficLight Violation at Location(x=68.271347...,NaN


In [74]:
display(df_simulation['speed_limit'].unique())

array([30., nan, 90., 60.])

In [75]:
# Not needed na - naging irrelevant bigla lol, pero putting it here just in case need natin
# GET LOCATION
def extract_location(text):
    if pd.isna(text):#
        return pd.Series([None, None])
    
    text = str(text)

    # Case 1: JSON-like
    if text.strip().startswith("{"):#
        try:
            data = json.loads(text)
            loc = data.get("location") or data.get("Location")
            if loc:
                return pd.Series([loc.get("x"), loc.get("y")])
        except:
            pass

    # Case 2: Plain text (CARLA format)
    match = re.search(r"Location.*?x[:=]\s*([\-\d.]+).*?y[:=]\s*([\-\d.]+)", text)
    if match:
        return pd.Series([float(match.group(1)), float(match.group(2))])

    return pd.Series([None, None])

df_simulation[["Location_X", "Location_Y"]] = df_simulation["details"].apply(extract_location)

In [76]:
# detect patterns
df_simulation["overspeed"] = df_simulation["details"].str.contains('"overspeed": true', na=False)
df_simulation["reduce_throttle"] = (
    (df_simulation["details"].str.contains("REDUCE_THROTTLE", na=False)) &
    (df_simulation["scenario"] == "SPEED_LIMIT")
)
df_simulation["resolved"] = df_simulation["details"].str.contains("VIOLATION_RESOLVED", na=False)

active = False
event_id = 0
event_ids = []

for overspeed, reduce_throttle, resolved in zip(
        df_simulation["overspeed"],
        df_simulation["reduce_throttle"],
        df_simulation["resolved"]):

    # start condition: active = False AND reduce_throttle OR overspeed
    if not active and (reduce_throttle or overspeed):
        active = True
        event_id += 1  # increment event_id for each new event

    # assign event_id if active
    if active:
        event_ids.append(event_id)
    else:
        event_ids.append(0)

    # end condition: resolved → stop counting
    if resolved:
        active = False

df_simulation["event_id"] = event_ids

In [77]:
df_simulation.columns

Index(['timestamp', 'participant_no', 'phase', 'scenario', 'speed_kmh',
       'event', 'details', 'speed_limit', 'Location_X', 'Location_Y',
       'overspeed', 'reduce_throttle', 'resolved', 'event_id'],
      dtype='object')

In [78]:
# checking lang 
display(df_simulation[df_simulation["scenario"].isin(["TRAFFIC_LIGHT"])].head(50))

,timestamp,participant_no,phase,scenario,speed_kmh,event,details,speed_limit,Location_X,Location_Y,overspeed,reduce_throttle,resolved,event_id
2440,1.771924e+09,17,4,TRAFFIC_LIGHT,16.78,YELLOW_LIGHT_PASS,"Passed yellow at Location(x=-58.246181, y=-2.3...",NaN,-58.246181,-2.379960,False,False,False,0
4155,1.771924e+09,17,4,TRAFFIC_LIGHT,16.85,YELLOW_LIGHT_PASS,"Passed yellow at Location(x=-19.716146, y=-197...",NaN,-19.716146,-197.653625,False,False,False,0
4458,1.771924e+09,17,4,TRAFFIC_LIGHT,7.09,RED_LIGHT_VIOLATION,TrafficLight Violation at Location(x=66.193649...,NaN,66.193649,-195.095139,False,False,False,0
4516,1.771924e+09,17,4,TRAFFIC_LIGHT,2.22,REACTION,action=NO_REACTION|time=3.01s|control={'thrott...,NaN,NaN,NaN,False,False,False,0
5052,1.771924e+09,17,4,TRAFFIC_LIGHT,16.56,YELLOW_LIGHT_PASS,"Passed yellow at Location(x=133.800629, y=-193...",NaN,133.800629,-193.991226,False,False,False,0
7489,1.771924e+09,17,4,TRAFFIC_LIGHT,35.28,RED_LIGHT_VIOLATION,TrafficLight Violation at Location(x=-57.89379...,NaN,-57.893791,130.662491,False,False,False,0
7549,1.771924e+09,17,4,TRAFFIC_LIGHT,36.42,REACTION,action=NO_REACTION|time=3.04s|control={'thrott...,NaN,NaN,NaN,False,False,False,0
8274,1.771924e+09,17,4,TRAFFIC_LIGHT,12.81,RED_LIGHT_VIOLATION,TrafficLight Violation at Location(x=-104.5033...,NaN,-104.503326,0.199147,False,False,False,0
8332,1.771924e+09,17,4,TRAFFIC_LIGHT,26.93,REACTION,action=NO_REACTION|time=3.03s|control={'thrott...,NaN,NaN,NaN,False,False,False,0
9536,1.771924e+09,17,1,TRAFFIC_LIGHT,14.79,YELLOW_LIGHT_PASS,"Passed yellow at Location(x=-85.075447, y=113....",NaN,-85.075447,113.654617,False,False,False,0


In [79]:
# Filter for SPEED_LIMIT scenario
speed_limit_df = df_simulation[df_simulation["scenario"] == "SPEED_LIMIT"]

# Count REDUCE_THROTTLE events
reduce_throttle_count = speed_limit_df["details"].str.contains("REDUCE_THROTTLE", na=False).sum()

# Count VIOLATION_RESOLVED events
violation_resolved_count = speed_limit_df["details"].str.contains("VIOLATION_RESOLVED", na=False).sum()

# Create a table to display
summary_table = pd.DataFrame({
    "Action": ["REDUCE_THROTTLE", "VIOLATION_RESOLVED"],
    "Count": [reduce_throttle_count, violation_resolved_count]
})

print(summary_table)

               Action  Count
0     REDUCE_THROTTLE    464
1  VIOLATION_RESOLVED    601


In [80]:
df_simulation.columns

Index(['timestamp', 'participant_no', 'phase', 'scenario', 'speed_kmh',
       'event', 'details', 'speed_limit', 'Location_X', 'Location_Y',
       'overspeed', 'reduce_throttle', 'resolved', 'event_id'],
      dtype='object')

### getting stop light coordinates

In [81]:
route1 = [
    (-5, -42), #0
    (-64, -2), #1
    (-85, 120), #2
    (-19, 138), #3
    (173, 80), #4
    (217, 63), #5
    (20, 193), #6
    (-77, 148), #7
    (-143, 8), #8
    (-100, -136),#9 
    (-14, -193), #10
    (-7, -91)#11
]

route2 = [
    (-8, -45),  #0
    (-12, 118), #1
    (-62, 131), #2
    (245, 81), #3
    (168, -207), #4
    (148, -149),  #5
    (148, -85),  #6
    (97, -77), #7
    (85, -120), #8
    (20, -138), #9
    (-8, -102)#10
]

route3 = [
    (-7, -41),   #0
    (-63, -2),   #1
    (-73, -122), #2
    (-15, -194), #3
    (69, -190),  #4
    (78,  -149), #5
    (74, -19),   #6
    (59, -4),    #7
    (216, 11),   #8
    (232, 42),   #9
    (28, 191),   #10
    (5, 149),    #11
    (172, 83),   #12
    (217, 63),   #13
    (246, -160), #14
    (166, -204), #15
    (146, -150), #16
    (97, -137),  #17
    (21, -137),  #18
    (-8, -102)   #19
]

route4 = [
    (-5, -42),  #0
    (-64, -2),  #1
    (-126, 42), #2
    (-142, 9),  #3
    (-100, -136),#4
    (-16, -195), #5
    (70, -195),  #6
    (138, -195), #7
    (221, -20),  #8
    (228, 41),   #9
    (187, 58),   #10
    (17, 128),   #11
    (-61, 129),  #12
    (-142, 9),   #13
    (-100, 0),   #14
    (-172, -122),#15 
    (-13, -131), #16
    (-5, -82)    #17
]

In [82]:
def nearest_point(route, x, y):
    min_dist = float("inf")
    nearest_idx = -1
    
    for i, (cx, cy) in enumerate(route):
        d = (x - cx)**2 + (y - cy)**2  # squared distance
        
        if d < min_dist:
            min_dist = d
            nearest_idx = i

    return nearest_idx, min_dist

In [83]:
df_simulation["nearest_idx"] = -1
df_simulation["distance_sq"] = np.nan
df_simulation["on_route"] = False

In [84]:
mask = df_simulation["phase"] == 1

for i in df_simulation[mask].index:
    x = df_simulation.at[i, "Location_X"]
    y = df_simulation.at[i, "Location_Y"]
    
    idx, dist = nearest_point(route1, x, y)
    
    df_simulation.at[i, "nearest_idx"] = idx
    df_simulation.at[i, "distance_sq"] = dist
    df_simulation.at[i, "on_route"] = dist <= 10**2

In [85]:
mask = df_simulation["phase"] == 2

for i in df_simulation[mask].index:
    x = df_simulation.at[i, "Location_X"]
    y = df_simulation.at[i, "Location_Y"]
    
    idx, dist = nearest_point(route2, x, y)
    
    df_simulation.at[i, "nearest_idx"] = idx
    df_simulation.at[i, "distance_sq"] = dist
    df_simulation.at[i, "on_route"] = dist <= 10**2

In [86]:
mask = df_simulation["phase"] == 3

for i in df_simulation[mask].index:
    x = df_simulation.at[i, "Location_X"]
    y = df_simulation.at[i, "Location_Y"]
    
    idx, dist = nearest_point(route3, x, y)
    
    df_simulation.at[i, "nearest_idx"] = idx
    df_simulation.at[i, "distance_sq"] = dist
    df_simulation.at[i, "on_route"] = dist <= 10**2

In [87]:
mask = df_simulation["phase"] == 4

for i in df_simulation[mask].index:
    x = df_simulation.at[i, "Location_X"]
    y = df_simulation.at[i, "Location_Y"]
    
    idx, dist = nearest_point(route4, x, y)
    
    df_simulation.at[i, "nearest_idx"] = idx
    df_simulation.at[i, "distance_sq"] = dist
    df_simulation.at[i, "on_route"] = dist <= 10**2

In [88]:
df_simulation['event_id'].max()

542

In [89]:
df_simulation[df_simulation["resolved"]].shape

(601, 17)

In [90]:
# Number of unique events started by REDUCE_THROTTLE in SPEED_LIMIT
df_simulation[df_simulation["reduce_throttle"]]["event_id"].nunique()

464

In [91]:
df_simulation[df_simulation["resolved"]] \
    .groupby("event_id") \
    .size() \
    .sort_values(ascending=False) \
    .head(25)

event_id
0      59
357     1
371     1
370     1
369     1
368     1
367     1
366     1
365     1
364     1
363     1
362     1
361     1
360     1
359     1
358     1
356     1
373     1
355     1
354     1
353     1
352     1
351     1
350     1
349     1
dtype: int64

In [92]:
df_simulation.columns

Index(['timestamp', 'participant_no', 'phase', 'scenario', 'speed_kmh',
       'event', 'details', 'speed_limit', 'Location_X', 'Location_Y',
       'overspeed', 'reduce_throttle', 'resolved', 'event_id', 'nearest_idx',
       'distance_sq', 'on_route'],
      dtype='object')

In [93]:
# checking lang if action = slow down and action = resolved has same event_id
df_simulation.loc[df_simulation["event_id"] == 90, ["speed_kmh","scenario", "speed_limit", "details", "event_id"]]

,speed_kmh,scenario,speed_limit,details,event_id
131868,37.33,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 37.33, ""speed_limit"": 30...",90
131869,37.77,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 37.77, ""speed_limit"": 30...",90
131870,37.77,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 37.77, ""speed_limit"": 30...",90
131871,37.77,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 37.77, ""speed_limit"": 30...",90
131872,38.34,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 38.34, ""speed_limit"": 30...",90
131873,38.53,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 38.53, ""speed_limit"": 30...",90
131874,38.89,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 38.89, ""speed_limit"": 30...",90
131875,39.03,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 39.03, ""speed_limit"": 30...",90
131876,39.03,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 39.03, ""speed_limit"": 30...",90
131877,39.03,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 39.03, ""speed_limit"": 30...",90


In [107]:
df_simulation.loc[
    (df_simulation["scenario"] == "SPEED_LIMIT") & 
    (df_simulation["phase"] == 1), 
    #(df_simulation["participant_no"]==16),
    ["phase","participant_no", "scenario", "Location_X", "Location_Y", "nearest_idx", "distance_sq", "on_route"]
].head(15)

,phase,participant_no,scenario,Location_X,Location_Y,nearest_idx,distance_sq,on_route
9533,1,17,SPEED_LIMIT,NaN,NaN,-1,inf,False
9534,1,17,SPEED_LIMIT,-85.912148,35.437183,1,1881.684901,False
9535,1,17,SPEED_LIMIT,NaN,NaN,-1,inf,False
9540,1,17,SPEED_LIMIT,NaN,NaN,-1,inf,False
9541,1,17,SPEED_LIMIT,-144.522446,-17.556885,8,655.472213,False
9542,1,17,SPEED_LIMIT,NaN,NaN,-1,inf,False
20403,1,18,SPEED_LIMIT,NaN,NaN,-1,inf,False
20404,1,18,SPEED_LIMIT,-89.194092,64.035187,2,3149.650702,False
20405,1,18,SPEED_LIMIT,NaN,NaN,-1,inf,False
20410,1,18,SPEED_LIMIT,NaN,NaN,-1,inf,False


In [95]:
df_simulation['event'].unique()

array(['WS_SEND', 'YELLOW_LIGHT_PASS', 'REACTION', 'SPEED_VIOLATION',
       'RED_LIGHT_VIOLATION', 'PHASE_STOP'], dtype=object)

In [96]:
df_simulation.to_csv("cleaned_Sim.csv", index=False)

# NASA TLX CLEANING

In [97]:
files = glob.glob("/Users/karencafe/Desktop/ThesisTest/NASACSV/*.csv")
# change to the path of the NASATLX folder containing all the csv files

df2 = pd.concat(
    (pd.read_csv(f) for f in files),
    ignore_index=True
)

display(df2)


,Participant,Task Id,Task Name,Mental Demand,Physical Demand,Temporal Demand,Performance,Effort,Frustration,Raw Score,MD Weight,PD Weight,TD Weight,PF Weight,EF Weight,FR Weight,Weighted Score
0,15,1,HO,15,5,60,15,10,5,18.333333,-1,-1,-1,-1,-1,-1,-1
1,15,2,AO,10,30,35,15,25,10,20.833333,-1,-1,-1,-1,-1,-1,-1
2,15,3,NC,10,30,25,10,30,5,18.333333,-1,-1,-1,-1,-1,-1,-1
3,15,4,AH,25,30,30,5,25,35,25.000000,-1,-1,-1,-1,-1,-1,-1
4,8,1,AH,70,60,40,70,60,80,63.333333,-1,-1,-1,-1,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,11,5,NC,5,15,5,65,50,0,23.333333,-1,-1,-1,-1,-1,-1,-1
76,5,1,AH,80,80,60,20,20,20,46.666667,-1,-1,-1,-1,-1,-1,-1
77,5,2,NC,80,80,60,10,10,10,41.666667,-1,-1,-1,-1,-1,-1,-1
78,5,3,HO,90,90,70,20,20,20,51.666667,-1,-1,-1,-1,-1,-1,-1


In [98]:
df2 = df2.drop(columns=["Task Id", "MD Weight", "PD Weight", "TD Weight", "PF Weight", "EF Weight", "FR Weight", "Weighted Score"])

display(df2)

,Participant,Task Name,Mental Demand,Physical Demand,Temporal Demand,Performance,Effort,Frustration,Raw Score
0,15,HO,15,5,60,15,10,5,18.333333
1,15,AO,10,30,35,15,25,10,20.833333
2,15,NC,10,30,25,10,30,5,18.333333
3,15,AH,25,30,30,5,25,35,25.000000
4,8,AH,70,60,40,70,60,80,63.333333
...,...,...,...,...,...,...,...,...,...
75,11,NC,5,15,5,65,50,0,23.333333
76,5,AH,80,80,60,20,20,20,46.666667
77,5,NC,80,80,60,10,10,10,41.666667
78,5,HO,90,90,70,20,20,20,51.666667


In [99]:
df2["Raw Score"] = df2["Raw Score"].round(2)
display(df2.head(3))

,Participant,Task Name,Mental Demand,Physical Demand,Temporal Demand,Performance,Effort,Frustration,Raw Score
0,15,HO,15,5,60,15,10,5,18.33
1,15,AO,10,30,35,15,25,10,20.83
2,15,NC,10,30,25,10,30,5,18.33


In [100]:
# Makes sure no mislabeled Task Name
print(df2['Task Name'].unique())

['HO' 'AO' 'NC' 'AH']


Clean task name 
- NC = no cue / baseline 
- AO = Audio Only 
- HO = Haptic Only 
- AH = Audio and Haptic / both Combined 

In [101]:
df2.columns = df2.columns.str.strip()          # removes hidden spaces
df2.columns = df2.columns.str.replace(" ", "_")

In [102]:
df2.to_csv("cleaned_NASATLX.csv", index=False)